In [1]:
# ==========================================
# STEP 1 : Import Libraries
# ==========================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import efficientnet_b0

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ==========================================
# Random Seed
# ==========================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ==========================================
# Device Configuration
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*60)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("="*60)

Device : cuda
GPU : Tesla T4


In [2]:
# ==========================================
# STEP 2 : Create DataFrame
# ==========================================

import os
import pandas as pd

# Dataset Path
dataset_path = "/kaggle/input/datasets/mstjebashazida/affectnet/archive (3)"

# Train and Test folders
folders = [
    os.path.join(dataset_path, "Train"),
    os.path.join(dataset_path, "Test")
]

image_paths = []
labels = []

# ==========================================
# Label Mapping (8 Classes)
# ==========================================

label_map = {
    "anger": 0,
    "contempt": 1,
    "disgust": 2,
    "fear": 3,
    "happy": 4,
    "neutral": 5,
    "sad": 6,
    "surprise": 7
}

# ==========================================
# Read Images
# ==========================================

for folder in folders:

    for emotion in os.listdir(folder):

        emotion_path = os.path.join(folder, emotion)

        if not os.path.isdir(emotion_path):
            continue

        emotion_name = emotion.lower()

        if emotion_name not in label_map:
            continue

        for img in os.listdir(emotion_path):

            img_path = os.path.join(emotion_path, img)

            image_paths.append(img_path)

            labels.append(label_map[emotion_name])

# ==========================================
# Create DataFrame
# ==========================================

df = pd.DataFrame({
    "image_path": image_paths,
    "label": labels
})

print("="*60)
print("Total Images :", len(df))
print("="*60)

print(df.head())

print("\nClass Distribution\n")
print(df["label"].value_counts().sort_index())

Total Images : 30626
                                          image_path  label
0  /kaggle/input/datasets/mstjebashazida/affectne...      7
1  /kaggle/input/datasets/mstjebashazida/affectne...      7
2  /kaggle/input/datasets/mstjebashazida/affectne...      7
3  /kaggle/input/datasets/mstjebashazida/affectne...      7
4  /kaggle/input/datasets/mstjebashazida/affectne...      7

Class Distribution

label
0    3218
1    2871
2    2477
3    3176
4    5044
5    5126
6    4675
7    4039
Name: count, dtype: int64


In [3]:
# ==========================================
# STEP 3 : 50:50 Train-Test Split
# ==========================================

from sklearn.model_selection import train_test_split

train_df_50, test_df_50 = train_test_split(
    df,
    test_size=0.50,
    stratify=df["label"],
    random_state=42,
    shuffle=True
)

# Reset Index
train_df_50 = train_df_50.reset_index(drop=True)
test_df_50 = test_df_50.reset_index(drop=True)

print("="*60)
print("50 : 50 Dataset Split")
print("="*60)

print("Training Images :", len(train_df_50))
print("Testing Images  :", len(test_df_50))
print("Total Images    :", len(df))

print("\nTraining Class Distribution")
print(train_df_50["label"].value_counts().sort_index())

print("\nTesting Class Distribution")
print(test_df_50["label"].value_counts().sort_index())

50 : 50 Dataset Split
Training Images : 15313
Testing Images  : 15313
Total Images    : 30626

Training Class Distribution
label
0    1609
1    1436
2    1238
3    1588
4    2522
5    2563
6    2337
7    2020
Name: count, dtype: int64

Testing Class Distribution
label
0    1609
1    1435
2    1239
3    1588
4    2522
5    2563
6    2338
7    2019
Name: count, dtype: int64


In [4]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ==========================
# Transforms
# ==========================

train_transform = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

# ==========================
# Dataset
# ==========================

class AffectNetDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.iloc[idx]["image_path"]
        label = int(self.df.iloc[idx]["label"])

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ==========================
# Dataset Objects
# ==========================

train_dataset_50 = AffectNetDataset(
    train_df_50,
    transform=train_transform
)

test_dataset_50 = AffectNetDataset(
    test_df_50,
    transform=test_transform
)

# ==========================
# DataLoaders
# ==========================

train_loader_50 = DataLoader(
    train_dataset_50,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader_50 = DataLoader(
    test_dataset_50,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train Images :", len(train_dataset_50))
print("Test Images :", len(test_dataset_50))

Train Images : 15313
Test Images : 15313


In [5]:
# ==========================================
# STEP 5 : Custom CNN Architecture (50:50)
# ==========================================

import torch
import torch.nn as nn

class CustomCNN(nn.Module):

    def __init__(self, num_classes=8):
        super(CustomCNN, self).__init__()

        # Feature Extraction
        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        # Classification Head
        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 12 * 12, 256),

            nn.ReLU(inplace=True),

            nn.Dropout(0.5),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# ==========================================
# Create CNN Model
# ==========================================

cnn_model_50 = CustomCNN(num_classes=8).to(device)

print(cnn_model_50)

CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=18432, out_features=256, bias=

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# Loss Function
# ==========================================
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# ==========================================
# Optimizer
# ==========================================
optimizer = optim.Adam(
    cnn_model_50.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

# ==========================================
# Learning Rate Scheduler
# ==========================================
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=80
)

# ==========================================
# Early Stopping
# ==========================================
class EarlyStopping:

    def __init__(self, patience=10):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def __call__(self, score):

        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True


early_stopping = EarlyStopping(patience=10)

epochs = 80

train_acc_history = []
test_acc_history = []

train_loss_history = []
test_loss_history = []

best_test_acc = 0

# ==========================================
# Training
# ==========================================
for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    cnn_model_50.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader_50:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = cnn_model_50(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader_50)

    # ---------------- VALIDATION ----------------
    cnn_model_50.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader_50:

            images = images.to(device)
            labels = labels.to(device)

            outputs = cnn_model_50(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    test_acc = 100 * correct / total
    test_loss = running_loss / len(test_loader_50)

    scheduler.step()

    # ==========================================
    # Save Best Model
    # ==========================================
    if test_acc > best_test_acc:

        best_test_acc = test_acc

        torch.save(
            cnn_model_50.state_dict(),
            "/kaggle/working/best_cnn_affectnet_50_50.pth"
        )

        print("✅ Best model updated.")

    train_acc_history.append(train_acc)
    test_acc_history.append(test_acc)

    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Test Accuracy  : {test_acc:.2f}%")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Test Loss      : {test_loss:.4f}")

    early_stopping(test_acc)

    if early_stopping.stop:

        print("\nEarly Stopping Triggered.")
        break

print("\n======================================================")
print(f"Best CNN Accuracy : {best_test_acc:.2f}%")
print("Saved Model : best_cnn_affectnet_50_50.pth")
print("======================================================")

✅ Best model updated.

Epoch 1/80
Train Accuracy : 27.28%
Test Accuracy  : 38.81%
Train Loss     : 1.8488
Test Loss      : 1.6446
✅ Best model updated.

Epoch 2/80
Train Accuracy : 33.12%
Test Accuracy  : 44.92%
Train Loss     : 1.7220
Test Loss      : 1.5715
✅ Best model updated.

Epoch 3/80
Train Accuracy : 34.52%
Test Accuracy  : 47.20%
Train Loss     : 1.7038
Test Loss      : 1.5553
✅ Best model updated.

Epoch 4/80
Train Accuracy : 36.79%
Test Accuracy  : 49.40%
Train Loss     : 1.6821
Test Loss      : 1.5023

Epoch 5/80
Train Accuracy : 37.28%
Test Accuracy  : 48.96%
Train Loss     : 1.6675
Test Loss      : 1.4736

Epoch 6/80
Train Accuracy : 37.72%
Test Accuracy  : 47.55%
Train Loss     : 1.6580
Test Loss      : 1.4816
✅ Best model updated.

Epoch 7/80
Train Accuracy : 37.86%
Test Accuracy  : 50.07%
Train Loss     : 1.6541
Test Loss      : 1.4816
✅ Best model updated.

Epoch 8/80
Train Accuracy : 38.55%
Test Accuracy  : 50.23%
Train Loss     : 1.6483
Test Loss      : 1.4694
✅ Be

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# ==========================================
# Load Best Saved CNN Model
# ==========================================

cnn_model_50 = CustomCNN(num_classes=8).to(device)

cnn_model_50.load_state_dict(
    torch.load(
        "/kaggle/working/best_cnn_affectnet_50_50.pth",
        map_location=device
    )
)

cnn_model_50.eval()

# ==========================================
# Extract Probabilities
# ==========================================

cnn_probs_50 = []
cnn_preds_50 = []
cnn_true_50 = []

with torch.no_grad():

    for images, labels in test_loader_50:

        images = images.to(device)

        outputs = cnn_model_50(images)

        probs = F.softmax(outputs, dim=1)

        cnn_probs_50.extend(probs.cpu().numpy())
        cnn_preds_50.extend(torch.argmax(probs, dim=1).cpu().numpy())
        cnn_true_50.extend(labels.numpy())

# ==========================================
# Convert to NumPy
# ==========================================

cnn_probs_50 = np.array(cnn_probs_50)
cnn_preds_50 = np.array(cnn_preds_50)
cnn_true_50 = np.array(cnn_true_50)

cnn_acc_50 = (cnn_preds_50 == cnn_true_50).mean() * 100

print("="*60)
print("CNN Probability Extraction Completed")
print("="*60)

print("Probabilities Shape :", cnn_probs_50.shape)
print("Predictions Shape   :", cnn_preds_50.shape)
print("True Labels Shape   :", cnn_true_50.shape)

print(f"\nCNN Accuracy : {cnn_acc_50:.2f}%")

In [ ]:
import torch
import torch.nn as nn

# ==========================================
# Bottleneck Block
# ==========================================

class Bottleneck(nn.Module):

    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):

        super(Bottleneck, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(
            out_channels,
            out_channels * self.expansion,
            kernel_size=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)

        self.downsample = downsample

    def forward(self, x):

        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        out = self.relu(out)

        return out

In [ ]:
import torch
import torch.nn as nn

# ==========================================
# ResNet50 Scratch
# ==========================================

class ResNet50Scratch(nn.Module):

    def __init__(self, num_classes=8):
        super(ResNet50Scratch, self).__init__()

        self.in_channels = 64

        # Stem
        self.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(64)

        self.relu = nn.ReLU(inplace=True)

        self.maxpool = nn.MaxPool2d(
            kernel_size=3,
            stride=2,
            padding=1
        )

        # Residual Layers
        self.layer1 = self._make_layer(
            Bottleneck,
            64,
            blocks=3,
            stride=1
        )

        self.layer2 = self._make_layer(
            Bottleneck,
            128,
            blocks=4,
            stride=2
        )

        self.layer3 = self._make_layer(
            Bottleneck,
            256,
            blocks=6,
            stride=2
        )

        self.layer4 = self._make_layer(
            Bottleneck,
            512,
            blocks=3,
            stride=2
        )

        # Pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))

        # Classification Layer
        self.fc = nn.Linear(
            512 * Bottleneck.expansion,
            num_classes
        )

    # ==========================================
    # Residual Layer
    # ==========================================
    def _make_layer(self, block, out_channels, blocks, stride):

        downsample = None

        if stride != 1 or self.in_channels != out_channels * block.expansion:

            downsample = nn.Sequential(

                nn.Conv2d(
                    self.in_channels,
                    out_channels * block.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(
                    out_channels * block.expansion
                )

            )

        layers = []

        layers.append(
            block(
                self.in_channels,
                out_channels,
                stride,
                downsample
            )
        )

        self.in_channels = out_channels * block.expansion

        for _ in range(1, blocks):

            layers.append(
                block(
                    self.in_channels,
                    out_channels
                )
            )

        return nn.Sequential(*layers)

    # ==========================================
    # Forward
    # ==========================================
    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

In [ ]:

# ==========================================
# STEP 8C : Create ResNet50 Model
# ==========================================

resnet_model_50 = ResNet50Scratch(num_classes=8).to(device)

print("=" * 60)
print("ResNet50 Model Created Successfully")
print("=" * 60)

print(resnet_model_50)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# Loss Function
# ==========================================

criterion = nn.CrossEntropyLoss()

# ==========================================
# Optimizer
# ==========================================

optimizer = optim.Adam(
    resnet_model_50.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ==========================================
# Learning Rate Scheduler
# ==========================================

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# ==========================================
# Early Stopping
# ==========================================

class EarlyStopping:

    def __init__(self, patience=8):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def __call__(self, score):

        if self.best_score is None or score > self.best_score:

            self.best_score = score
            self.counter = 0

        else:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True


early_stopping = EarlyStopping(patience=8)

epochs = 60

train_acc_history_res = []
test_acc_history_res = []

train_loss_history_res = []
test_loss_history_res = []

best_test_acc = 0

# ==========================================
# Training
# ==========================================

for epoch in range(epochs):

    # ---------------- TRAIN ----------------

    resnet_model_50.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader_50:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet_model_50(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader_50)

    # ---------------- VALIDATION ----------------

    resnet_model_50.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader_50:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet_model_50(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    test_acc = 100 * correct / total
    test_loss = running_loss / len(test_loader_50)

    scheduler.step(test_loss)

    # ==========================================
    # Save Best Model
    # ==========================================

    if test_acc > best_test_acc:

        best_test_acc = test_acc

        torch.save(
            resnet_model_50.state_dict(),
            "/kaggle/working/best_resnet50_affectnet_50_50.pth"
        )

        print("✅ Best model updated.")

    train_acc_history_res.append(train_acc)
    test_acc_history_res.append(test_acc)

    train_loss_history_res.append(train_loss)
    test_loss_history_res.append(test_loss)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Test Accuracy  : {test_acc:.2f}%")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Test Loss      : {test_loss:.4f}")

    early_stopping(test_acc)

    if early_stopping.stop:

        print("\nEarly Stopping Triggered.")
        break

print("\n====================================================")
print(f"Best ResNet50 Accuracy : {best_test_acc:.2f}%")
print("Saved Model : best_resnet50_affectnet_50_50.pth")
print("====================================================")

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# ==========================================
# Load Best Saved ResNet50 Model
# ==========================================

resnet_model_50 = ResNet50Scratch(num_classes=8).to(device)

resnet_model_50.load_state_dict(
    torch.load(
        "/kaggle/working/best_resnet50_affectnet_50_50.pth",
        map_location=device
    )
)

resnet_model_50.eval()

# ==========================================
# Extract Probabilities
# ==========================================

resnet_probs_50 = []
resnet_preds_50 = []
resnet_true_50 = []

with torch.no_grad():

    for images, labels in test_loader_50:

        images = images.to(device)

        outputs = resnet_model_50(images)

        probs = F.softmax(outputs, dim=1)

        resnet_probs_50.extend(probs.cpu().numpy())

        resnet_preds_50.extend(
            torch.argmax(probs, dim=1).cpu().numpy()
        )

        resnet_true_50.extend(labels.numpy())

# ==========================================
# Convert to NumPy Arrays
# ==========================================

resnet_probs_50 = np.array(resnet_probs_50)

resnet_preds_50 = np.array(resnet_preds_50)

resnet_true_50 = np.array(resnet_true_50)

resnet_acc_50 = (
    resnet_preds_50 == resnet_true_50
).mean() * 100

print("="*60)
print("ResNet50 Probability Extraction Completed")
print("="*60)

print("Probabilities Shape :", resnet_probs_50.shape)
print("Predictions Shape   :", resnet_preds_50.shape)
print("True Labels Shape   :", resnet_true_50.shape)

print(f"\nResNet50 Accuracy : {resnet_acc_50:.2f}%")

In [ ]:
# ==========================================
# STEP 11A : EfficientNet DataLoader (50:50)
# ==========================================

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ==========================================
# EfficientNet Transforms
# ==========================================

eff_train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

eff_test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ==========================================
# Dataset
# ==========================================

class AffectNetEfficientDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.iloc[idx]["image_path"]

        label = int(self.df.iloc[idx]["label"])

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# ==========================================
# Train Dataset
# ==========================================

eff_train_dataset_50 = AffectNetEfficientDataset(
    train_df_50,
    transform=eff_train_transform
)

# ==========================================
# Test Dataset
# ==========================================

eff_test_dataset_50 = AffectNetEfficientDataset(
    test_df_50,
    transform=eff_test_transform
)

# ==========================================
# DataLoaders
# ==========================================

eff_train_loader_50 = DataLoader(
    eff_train_dataset_50,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

eff_test_loader_50 = DataLoader(
    eff_test_dataset_50,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("="*60)
print("EfficientNet DataLoader Ready")
print("="*60)

print("Training Images :", len(eff_train_dataset_50))
print("Testing Images  :", len(eff_test_dataset_50))

images, labels = next(iter(eff_train_loader_50))

print("\nBatch Shape :", images.shape)
print("Labels Shape:", labels.shape)

In [ ]:
# ==========================================
# STEP 11B : EfficientNet-B0 Architecture
# ==========================================

import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# ==========================================
# Load Pretrained EfficientNet-B0
# ==========================================

efficientnet_model_50 = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

# ==========================================
# Replace Classification Layer
# ==========================================

in_features = efficientnet_model_50.classifier[1].in_features

efficientnet_model_50.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 8)   # 8 emotion classes
)

# ==========================================
# Move Model to GPU
# ==========================================

efficientnet_model_50 = efficientnet_model_50.to(device)

print("="*60)
print("EfficientNet-B0 Model Created Successfully")
print("="*60)

print(efficientnet_model_50)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# Loss Function
# ==========================================
criterion = nn.CrossEntropyLoss()

# ==========================================
# Optimizer
# ==========================================
optimizer = optim.Adam(
    efficientnet_model_50.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ==========================================
# Scheduler
# ==========================================
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

# ==========================================
# Early Stopping
# ==========================================
class EarlyStopping:

    def __init__(self, patience=8):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def __call__(self, score):

        if self.best_score is None or score > self.best_score:

            self.best_score = score
            self.counter = 0

        else:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True


early = EarlyStopping(patience=8)

epochs = 60

train_acc_list_eff = []
test_acc_list_eff = []

train_loss_list_eff = []
test_loss_list_eff = []

best_test_acc = 0

# ==========================================
# Training Loop
# ==========================================
for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    efficientnet_model_50.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in eff_train_loader_50:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = efficientnet_model_50(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(eff_train_loader_50)

    # ---------------- VALIDATION ----------------
    efficientnet_model_50.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in eff_test_loader_50:

            images = images.to(device)
            labels = labels.to(device)

            outputs = efficientnet_model_50(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    test_acc = 100 * correct / total
    test_loss = running_loss / len(eff_test_loader_50)

    scheduler.step(test_loss)

    # ==========================================
    # Save Best Model
    # ==========================================
    if test_acc > best_test_acc:

        best_test_acc = test_acc

        torch.save(
            efficientnet_model_50.state_dict(),
            "/kaggle/working/best_efficientnet_affectnet_50_50.pth"
        )

        print("✅ Best model saved.")

    train_acc_list_eff.append(train_acc)
    test_acc_list_eff.append(test_acc)

    train_loss_list_eff.append(train_loss)
    test_loss_list_eff.append(test_loss)

    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"Train Accuracy : {train_acc:.2f}%")
    print(f"Test Accuracy  : {test_acc:.2f}%")
    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Test Loss      : {test_loss:.4f}")

    early(test_acc)

    if early.stop:

        print("\nEarly stopping triggered.")
        break

print("="*60)
print(f"Best EfficientNet Accuracy : {best_test_acc:.2f}%")
print("Saved Model : /kaggle/working/best_efficientnet_affectnet_50_50.pth")
print("="*60)